In [ ]:
import pandas as pd
import re
from datetime import datetime
import wikibaseintegrator
from wikibaseintegrator import WikibaseIntegrator, wbi_helpers, wbi_login, datatypes
from wikibaseintegrator.wbi_config import config as wbi_config
from wikibaseintegrator.entities import ItemEntity
from wikibaseintegrator.models import Qualifiers, References, Reference
from wikibaseintegrator.wbi_enums import ActionIfExists
import logging
import json
import random
from pathlib import Path
from typing import Optional, Union
from dataclasses import dataclass, field
from __future__ import annotations
from enum import Enum
import numpy as np
from time import sleep

collections_dir = Path("../wikidata/metadata_collections/")

with Path('../authorization.json').open(mode='r') as authorization_file:
    authorization = json.load(authorization_file)

USER_AUTH = authorization['user_auth']
USERNAME = authorization['username']
PASSWORD = authorization['password']
CONSUMER_TOKEN = authorization['consumer_token']
CONSUMER_SECRET = authorization['consumer_secret']

BOTNAME = authorization['botname']
    

logging.basicConfig(filename='Mass_Upload.log', force=True,
                    format='%(asctime)s %(message)s', 
                    datefmt='%Y/%m/%d %H:%M:%S',
                    encoding='utf-8', 
                    level=logging.DEBUG)

logger = logging.getLogger('Make-Items')
logger.debug('Start logging')


wbi_config['USER_AGENT'] = f'{BOTNAME} (https://www.wikidata.org/wiki/User:{USERNAME})'
wbi_config['MEDIAWIKI_API_URL'] = 'https://test.wikidata.org/w/api.php'


PROPS = {'instance_of':'P31',
        'author':'P50',
         'title':'P1476',
         'has_edition':'P747',
         'edition_of': 'P629',
         'based_on' : 'P144',
         'language':'P407',
         'publication_date':'P577',
         'work_available_at_URL':'P953',
         'has_edition_or_translation' : 'P747',
         'project_gb_ebook_id' : 'P2034',
         'sex_or_gender' :'P21',
         'family_name' : 'P734',
         'given_name' : 'P735',
         'date_of_birth' : 'P569',
         'date_of_death' : 'P570'
         }
ENTITIES = {
   'literary_work':'Q7725634', 
   'edition' : 'Q3331189',
   'German':'Q188',
   'Hugo_Ball':'Q70989'
}

In [ ]:
metadata = pd.read_csv(Path(collections_dir, 'de_fiction_metadata_2025-11-14T18.csv'), 
                       index_col=0, dtype={'gutenberg_id':str})
metadata

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,filename,author_gender,num_sents,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,1e_qid_by
index,,,,,,,,,,,,,,,,,,,,,
0,Arthur Achleitner,Achleitner,Arthur,Der Finanzer,NaN,NaN,1916,https://www.projekt-gutenberg.org/achleitn/fin...,achleitn,Q77439,...,Arthur_Achleitner_-_Der_Finanzer.txt,m,1723,24034,PG-DE,NaN,NaN,NaN,NaN,NaN
1,Arthur Achleitner,Achleitner,Arthur,Das Schloß im Moor,NaN,NaN,1903,https://www.projekt-gutenberg.org/achleitn/moo...,achleitn,Q77439,...,Arthur_Achleitner_-_Das_Schloß_im_Moor.txt,m,3460,51072,PG-DE,NaN,NaN,NaN,NaN,NaN
2,Arthur Achleitner,Achleitner,Arthur,Familie Lugmüller,NaN,NaN,1896,https://www.projekt-gutenberg.org/achleitn/lug...,achleitn,Q77439,...,Arthur_Achleitner_-_Familie_Lugmüller.txt,m,1967,26741,PG-DE,NaN,NaN,NaN,NaN,NaN
3,Arthur Achleitner,Achleitner,Arthur,Der Bezirkshauptmann. Erster Teil,NaN,NaN,1901,https://www.projekt-gutenberg.org/achleitn/bez...,achleitn,Q77439,...,Arthur_Achleitner_-_Der_Bezirkshauptmann._Erst...,m,2614,34895,PG-DE,NaN,NaN,NaN,NaN,NaN
4,Arthur Achleitner,Achleitner,Arthur,Geschichten aus den Bergen,NaN,NaN,1910,https://www.projekt-gutenberg.org/achleitn/ber...,achleitn,Q77439,...,Arthur_Achleitner_-_Geschichten_aus_den_Bergen...,m,6350,127192,PG-DE,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4229,Arnold Zweig,Zweig,Arnold,Die Novellen um Claudia,NaN,52478,1912,NaN,NaN,NaN,...,"Zweig,_Arnold_-_Die_Novellen_um_Claudia-52478.txt",m,2720,47335,PG-US,NaN,NaN,NaN,NaN,NaN
4230,Friderike Maria Burger Winternitz Zweig,Zweig,Friderike Maria Burger Winternitz,Vögelchen,NaN,57114,1919,NaN,NaN,NaN,...,"Zweig,_Friderike_Maria_Burger_Winternitz_-_Vög...",m,7846,104521,PG-US,NaN,NaN,NaN,NaN,NaN
4231,Stefan Zweig,Zweig,Stefan,Amok: Novellen einer Leidenschaft,Q675609,57850,1922,NaN,zweig,Q78491,...,"Zweig,_Stefan_-_Amok:_Novellen_einer_Leidensch...",m,3087,69716,PG-US,NaN,NaN,NaN,NaN,NaN


In [ ]:
author_qids = set(metadata['author_qid'].dropna())
print(len(author_qids))
author_qids

In [ ]:
def query_works(author_qids : list[str]):
    query = """SELECT ?author_qid ?author_qidLabel ?work_qid ?work_qidLabel ?title ?title_de ?work_alt ?publication_date ?language
        WHERE
        {
        VALUES ?author_qid { wd:""" + " wd:".join(author_qids) + """}
        ?work_qid wdt:P31 wd:Q7725634 .
        ?work_qid wdt:P50 ?author_qid .
        OPTIONAL {
            ?work_label rdfs:label ?work_qid .
            }
        OPTIONAL {
            ?work_qid skos:altLabel ?work_alt FILTER (lang(?work_alt) = "de") .
            }
        OPTIONAL {
            ?work_qid wdt:P1476 ?title . 
            }
        OPTIONAL {
            ?work_qid wdt:P1476 ?title_de FILTER (lang(?title_de) = "de") . 
            }
        OPTIONAL {
            ?work_qid wdt:P577 ?publication_date .
            }
            OPTIONAL {
            ?work_qid wdt:P407 ?language .
            }
        SERVICE wikibase:label { bd:serviceParam wikibase:language "de,mul,en". }
        }
        ORDER BY desc(?author_qidLabel) desc(?title)"""
    return wbi_helpers.execute_sparql_query(query, 
                                            user_agent=wbi_config['USER_AGENT'])

# Here's how to use it:
works_raw = query_works(['Q70989'])
works_raw['head']

In [19]:
def qid_from_url(url : str):
    return re.search(r'Q\d+$',url).group()

def chunker_alt(seq, size):
    return [seq[pos:pos + size] for pos in range(0, len(seq), size)]

def chunk_lengths(total_sents, num_chunks):
  base_chunk_size, remainder = divmod(total_sents, num_chunks)
  return [base_chunk_size + 1] * remainder + [base_chunk_size] * (num_chunks - remainder)

def chunker(seq, size = None, num_chunks = None, start_at = None, stop_before = None, ind = None, return_ind = False):
  if num_chunks == None and size == None:
    raise RuntimeError("One of num_chunks and size must be given!")

  if (num_chunks and (len(seq) < num_chunks)) or (size and (len(seq) < size)):
     raise RuntimeError("sequence is too short to be chunked in this way")
  #make chunks of fixed size if possible 
  if (size != None) and (num_chunks == None):
    all_chunks = [seq[pos:pos + size] for pos in range(0, len(seq), size)]
  
  #guarantee certain number of chunks
  elif (size == None) and (num_chunks != None):
    lengths = chunk_lengths(len(seq), num_chunks)
    sums = [sum(lengths[0:i]) for i in range(0, num_chunks + 1)]
    chunk_borders = zip(sums[:-1], sums[1:])
    all_chunks = [seq[start:end] for start, end in chunk_borders]
  else:
    raise RuntimeError("chunker should not be called with both size and num_chunks args")
  
  # If only part requested...
  if start_at == None:
    start_at = 0
  if stop_before == None:
    stop_before = len(all_chunks)
  if ind == None:
     ind = range(start_at, stop_before)
  else:
     ind = ind[start_at:stop_before]
     
  chunks = [all_chunks[i] for i in ind if i < len(all_chunks)]  
  #Finally, return
  if return_ind:
    return chunks, ind
  else:
    return chunks

def works_by_author_qids(author_qids : list[str], limit_nr_qids_per_request = 300, time_out=60):
    if len(author_qids) < limit_nr_qids_per_request:
       author_qids = [author_qids]
    else:
       author_qids = chunker_alt(seq=author_qids, size=limit_nr_qids_per_request)

    works_by_authors = {}

    for author_qids_part in author_qids: 
      works_raw = query_works(author_qids_part)
      vars = works_raw['head']['vars']

      for n, work in enumerate(works_raw['results']['bindings']):
          work_gist = {}
          for label in vars:
              if label in work:
                  if ('language' == label) or ('qid' in label) and not ('Label' in label):
                      work_gist[label] = qid_from_url(work[label]['value'])
                  else:
                      work_gist[label] = work[label]['value']
              else:
                  print(f'{label} not in item nr {n}: {work}')
          if 'author_qid' in work_gist:
              author_qid = work_gist['author_qid']
              if author_qid not in works_by_authors:
                  works_by_authors[author_qid] = []
              works_by_authors[author_qid].append(work_gist)
      sleep(time_out)
    return works_by_authors

In [14]:
chunker(range(6), size=4)

[range(0, 4), range(4, 6)]

In [17]:
chunker_alt(range(3), size=4)

[range(0, 3)]

In [ ]:
works_by_authors = works_by_author_qids(list(author_qids))
works_by_authors

In [52]:
authors = pd.read_csv(Path(collections_dir, 'de_fiction_authors_metadata-2025-11-15T21:07.csv'), index_col=0)
authors = authors.assign(no_works=np.nan).astype({'no_works':object})
authors = authors.reset_index()
authors.index = authors['author']
authors = authors.drop(columns=['author'])
authors

,index,author_first,author_last,author_gender,pgde_author_id,author_qid,num_sents,num_tokens,works,author_qids,urls,work_qids,sources,no_works
author,,,,,,,,,,,,,,
A. K. Ruh,0,A. K.,Ruh,f,NaN,NaN,3825,49845,1,0,0,1,1,NaN
Abraham Manuel Fröhlich,1,Abraham Manuel,Fröhlich,m,froehlia,Q115992,1600,25236,1,1,1,0,1,NaN
Achim von Arnim,2,Achim von,Arnim,m,arnim,Q70988,28209,588632,8,1,2,2,1,NaN
Ada Christen,3,Ada,Christen,f,NaN,NaN,1719,36651,1,0,0,0,1,NaN
Adalbert Stifter,4,Adalbert,Stifter,m,stifter,Q168542,49222,1075301,20,1,20,4,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Wolf Graf von Baudissin,1060,Wolf Graf von,Baudissin,m,baudiswg,Q98142,7509,172869,3,1,3,0,1,NaN
Wolf von Baudissin,1061,Wolf von,Baudissin,m,NaN,NaN,2915,70463,1,0,0,0,1,NaN
Wolfgang Hellmert,1062,Wolfgang,Hellmert,m,hellmert,Q23710233,1603,19429,1,1,1,0,1,NaN


In [ ]:
def no_case_punct(s : str):
    return re.sub(r'[^\w\s]', '', s).lower()


class MatchType(Enum):
    NO_MATCH = 0
    EXACT = 20
    NO_CASE_PUNCT = 19
    SUBSTR_MIDDLE = 8
    SUBSTR_END = 9
    SUBSTR_START = 10



    def __lt__(self, m : MatchType):
        return self.value < m.value
    
    def __float__(self):             # I thought this was for np.isna()
        return float(self.value)     # Well, if it was, that doesn't work


@dataclass
class MatchRecord:
    index : int         #index in our table
    type : MatchType
    wd_qid : str = ''
    field : str =''
    score : float = -1

# MAKE THIS GIANT LOOP A FUNCTION

matches = {}

metadata_authors_with_qid = metadata[metadata['author_qid'].notna()]

for i, author_qid, author_name, title, pubyear, work_qid in zip(metadata_authors_with_qid.index, 
                                                                metadata_authors_with_qid['author_qid'], 
                                                             metadata_authors_with_qid['author'], 
                                                metadata_authors_with_qid['title'], 
                                                metadata_authors_with_qid['year'], 
                                                metadata_authors_with_qid['work_qid']):
    if (author_qid in works_by_authors):
        #print(title)
        #print(author_qid)
        work_matches = []
        for work in works_by_authors[author_qid]:
            for field in ['title_de', 'title', 'work_qidLabel']:
                match_type = MatchType.NO_MATCH
                if (field in work):
                    #print(f'{title} ?= {work[field]}')
                    work_field= work[field]
                    if title == work_field:
                        msg = f'Exact match to {field}: {i} {title} by {author_name} with {work_qid}'
                        match_type = MatchType.EXACT
                        break # if this work matches our row, no need to look at other fields
                    
                    title_ncnp = no_case_punct(title)
                    work_field_ncnp = no_case_punct(work_field)
                    if title_ncnp == work_field_ncnp:
                        msg = f'No case/punct to {field}: {i} {title_ncnp} by {author_name} with {work_qid}'
                        match_type = MatchType.NO_CASE_PUNCT
                        break # if this work matches our row, no need to look at other fields
                    # Check for matches at beginning 
                    if re.match('^' + title_ncnp + '.*', work_field_ncnp) or re.match('^' + work_field_ncnp + '.*', title_ncnp):
                        msg = f'String match at start to {field}: {i} {title_ncnp} by {author_name} with {work_qid}'
                        logger.info(msg)
                        match_type = MatchType.SUBSTR_START
                        # This time, I don't think we should break
                    elif re.match('.*' + title_ncnp + '$', work_field_ncnp) or re.match('.*' + work_field_ncnp + '$', title_ncnp):
                        msg = f'String match at end to {field}: {i} {title_ncnp} by {author_name} with {work_qid}'
                        logger.info(msg)
                        match_type = MatchType.SUBSTR_START
                        # This time, I don't think we should break
                    elif (title_ncnp in work_field) or (work_field_ncnp in title):
                        msg = f'Substring match in middle to {field}: {i} {title_ncnp} by {author_name} with {work_qid}'
                        logger.info(msg)
                        match_type = MatchType.SUBSTR_MIDDLE
                    # SHOULD COMPUTE A FUZZY MATCHING METRIC HERE
            if match_type != MatchType.NO_MATCH:
                mrec = MatchRecord(index = i, wd_qid = work['work_qid'], type=match_type, field=field)
                print(msg)
            else:
                mrec = MatchRecord(index = i, type = match_type)
            work_matches.append(mrec)
        matches[i] = work_matches
                    
    else:
        # What does this actually mean: We have a qid for the author, so they exist on wikidata
        # But they do not seem to have any works associated to them
        # Worth investigating who they are!
        msg = f'Potential author without literary works:  {author_qid} ({author_name})'
        print(msg)
        logger.info(msg)
        authors.at[author_name, 'no_works'] = True
        # No need to repeat for this author!
        continue

In [54]:
no_works_authors = authors[authors['no_works'].notna() & (authors['no_works'].notna() == True)].sort_values(['author_gender', 'author_last'])
no_works_authors

,index,author_first,author_last,author_gender,pgde_author_id,author_qid,num_sents,num_tokens,works,author_qids,urls,work_qids,sources,no_works
author,,,,,,,,,,,,,,
Luise Ahlborn,762,Luise,Ahlborn,f,ahlborn,Q1324050,5199,77373,1,1,1,0,1,True
Brigitte Augusti,141,Brigitte,Augusti,f,augusti,Q19144479,2301,43187,1,1,1,0,1,True
Käthe van Beeker,716,Käthe van,Beeker,f,beeker,Q1795559,3724,62196,1,1,1,0,1,True
Alice Berend,61,Alice,Berend,f,berend,Q2646840,13158,153700,4,1,3,0,1,True
Ida Bindschedler,552,Ida,Bindschedler,f,bindsche,Q1656493,5718,68745,1,1,1,0,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Paul Zifferer,900,Paul,Zifferer,m,zifferer,Q3372442,1740,38777,1,1,1,0,1,True
Ignaz Zingerle,558,Ignaz,Zingerle,m,zingerle,Q873784,1872,30443,1,1,1,0,1,True
Fedor von Zobeltitz,300,Fedor von,Zobeltitz,m,zobeltit,Q100791,34404,435803,5,1,4,0,2,True


In [55]:
summary = no_works_authors.drop(columns=['index', 'author_first', 'author_last', 'sources']).groupby('author_gender').sum()
summary['num_authors'] = no_works_authors.reset_index().groupby('author_gender')['author'].nunique()
summary = summary[['num_authors', 'author_qids', 
                   'works', 'work_qids', 'num_tokens', 
                   'num_sents', 'urls']].rename(columns={'num_tokens' : 'tokens', 'num_sents' : 'sents'})
summary

,num_authors,author_qids,works,work_qids,tokens,sents,urls
author_gender,,,,,,,
f,70,70,217,1,12098115,768426,182
m,319,319,1119,1,62752413,3984839,1003


In [56]:
author_qids_f_maybe_no_works = no_works_authors[no_works_authors['author_gender'] == 'f']['author_qid']
author_qids_f_maybe_no_works

author
Luise Ahlborn          Q1324050
Brigitte Augusti      Q19144479
Käthe van Beeker       Q1795559
Alice Berend           Q2646840
Ida Bindschedler       Q1656493
                        ...    
Ottilie Wildermuth       Q69311
Olga Wohlbrück         Q2019750
Anny Wothe              Q567448
Sophie Wörishöffer       Q75259
Marie Zedelius        Q19224537
Name: author_qid, Length: 70, dtype: object

In [ ]:
"wd:" + " wd:".join(author_qids_f_maybe_no_works.values)

The following verifies that these authors really all have no works.

In [67]:
query = '''SELECT ?author_qid ?author_qidLabel ?sex_or_genderLabel (COUNT(?work) AS ?work_count)
WHERE {
  VALUES ?author_qid {
  wd:''' + " wd:".join(author_qids_f_maybe_no_works.values) + '''
  }
  # gender
  OPTIONAL { ?author_qid wdt:P21 ?sex_or_gender . }
  # works 
  OPTIONAL {
    ?work wdt:P50 ?author_qid .
    ?work wdt:P31 wd:Q7725634 .
  }
  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en" .
  }
}
GROUP BY ?author_qid ?author_qidLabel ?sex_or_genderLabel
ORDER BY ?sex_or_genderLabel ?author_qidLabel ?author_qid'''

In [ ]:
result = wbi_helpers.execute_sparql_query(query, 
                                          user_agent=wbi_config['USER_AGENT'])
result

In [73]:
has_works = [record['author_qid']['value'] for record in result['results']['bindings'] 
             if record['work_count']['value'] != '0']
has_works

[]

In [85]:
author_qids_no_works = [author_qid for author_qid in author_qids_f_maybe_no_works 
               if not author_qid in has_works]
len(author_qids_no_works)

70

# We must verify we don't have any works with multiple authors:

In [86]:
metadata[metadata['author_qid'].isin(author_qids_no_works)]

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,filename,author_gender,num_sents,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,1e_qid_by
index,,,,,,,,,,,,,,,,,,,,,
19,Luise Ahlborn,Ahlborn,Luise,Schloß Favorite,NaN,NaN,1887,https://www.projekt-gutenberg.org/ahlborn/favo...,ahlborn,Q1324050,...,Luise_Ahlborn_-_Schloß_Favorite.txt,f,5199,77373,PG-DE,NaN,NaN,NaN,NaN,NaN
124,Brigitte Augusti,Augusti,Brigitte,Mädchenlose,NaN,NaN,1888,https://www.projekt-gutenberg.org/augusti/maed...,augusti,Q19144479,...,Brigitte_Augusti_-_Mädchenlose.txt,f,2301,43187,PG-DE,NaN,NaN,NaN,NaN,NaN
192,Käthe van Beeker,Beeker,Käthe van,Der Ring der Nuramaja,NaN,NaN,1919,https://www.projekt-gutenberg.org/beeker/ringn...,beeker,Q1795559,...,Käthe_van_Beeker_-_Der_Ring_der_Nuramaja.txt,f,3724,62196,PG-DE,NaN,NaN,NaN,NaN,NaN
197,Alice Berend,Berend,Alice,Frau Hempels Tochter,NaN,NaN,1912,NaN,NaN,Q2646840,...,Alice_Berend_-_Frau_Hempels_Tochter.txt,f,3286,47754,PG-DE,NaN,NaN,NaN,NaN,NaN
198,Alice Berend,Berend,Alice,Die Bräutigame der Babette Bomberling,NaN,NaN,1915,https://www.projekt-gutenberg.org/berend/bombe...,berend,Q2646840,...,Alice_Berend_-_Die_Bräutigame_der_Babette_Bomb...,f,2891,32898,PG-DE,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4130,Anna Schieber,Schieber,Anna,Alle guten Geister...: Roman,NaN,54469,1905,NaN,NaN,Q563171,...,"Schieber,_Anna_-_Alle_guten_Geister...:_Roman-...",f,9418,121841,PG-US,NaN,NaN,NaN,NaN,NaN
4131,Anna Schieber,Schieber,Anna,Heimat: Erzählungen,NaN,67701,1915,NaN,NaN,Q563171,...,"Schieber,_Anna_-_Heimat:_Erzählungen-67701.txt",f,2533,37457,PG-US,NaN,NaN,NaN,NaN,NaN
4132,Anna Schieber,Schieber,Anna,Ludwig Fugeler: Roman,NaN,34424,1918,NaN,NaN,Q563171,...,"Schieber,_Anna_-_Ludwig_Fugeler:_Roman-34424.txt",f,3988,98317,PG-US,NaN,NaN,NaN,NaN,NaN


In [87]:
grouped = metadata[metadata['author_qid'].isin(author_qids_no_works)].groupby('title')
by_title = grouped['author'].nunique()
print(by_title.max())
by_title

1


title
... und hätte der Liebe nicht              1
Alle guten Geister...: Roman               1
Als der Mond in Dorothees Zimmer schien    1
Alte Liebe und anderes                     1
Am Glück vorbei                            1
                                          ..
Warenhaus Groß & Comp.                     1
Was bin ich dir?                           1
Wenn Mütter sündigen ...                   1
Winkelquartett                             1
Wolfsburg                                  1
Name: author, Length: 217, dtype: int64

## Deal with parts of works

In [198]:
view = metadata[metadata['author_qid'].isin(author_qids_no_works)]
select_part_of_works = view['title'].str.contains('Band', flags=re.IGNORECASE) \
    | view['title'].str.contains('Teil', flags=re.IGNORECASE) \
     | view['title'].str.contains('Theil', flags=re.IGNORECASE) \
        | view['title'].str.contains('Buch', flags=re.IGNORECASE) \
            | view['title'].str.contains('1', flags=re.IGNORECASE) \
            | view['title'].str.contains('2', flags=re.IGNORECASE) \
            | view['title'].str.contains('3 ', flags=re.IGNORECASE) \
             | view['title'].str.endswith(' I') \
            | view['title'].str.contains(' II', flags=re.IGNORECASE) \
            | view['title'].str.contains(' III', flags=re.IGNORECASE)

view[select_part_of_works]

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,1e_qid_by,work_qid_date,ed1_qid_date,pgde_qid_date
index,,,,,,,,,,,,,,,,,,,,,
650,Nataly von Eschstruth,Eschstruth,Nataly von,Der Majoratsherr. I. Band,NaN,NaN,1898,https://www.projekt-gutenberg.org/eschstru/maj...,eschstru,Q87140,...,53790,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:29,2025-11-16T00:46:29,2025-11-16T00:46:29
653,Nataly von Eschstruth,Eschstruth,Nataly von,Frieden II,NaN,NaN,1905,NaN,NaN,Q87140,...,52244,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:31,2025-11-16T00:46:31,2025-11-16T00:46:31
654,Nataly von Eschstruth,Eschstruth,Nataly von,Frühlingsstürme. Band II,NaN,NaN,1899,https://www.projekt-gutenberg.org/eschstru/fru...,eschstru,Q87140,...,52042,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:32,2025-11-16T00:46:32,2025-11-16T00:46:32
656,Nataly von Eschstruth,Eschstruth,Nataly von,Jung gefreit - 1,NaN,NaN,1897,NaN,NaN,Q87140,...,61541,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:33,2025-11-16T00:46:33,2025-11-16T00:46:33
657,Nataly von Eschstruth,Eschstruth,Nataly von,Polnisch Blut. Erster Band<,NaN,NaN,1887,NaN,NaN,Q87140,...,57601,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:34,2025-11-16T00:46:34,2025-11-16T00:46:34
659,Nataly von Eschstruth,Eschstruth,Nataly von,Hazard. Erster Band,NaN,NaN,1888,https://www.projekt-gutenberg.org/eschstru/haz...,eschstru,Q87140,...,58351,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:36,2025-11-16T00:46:36,2025-11-16T00:46:36
660,Nataly von Eschstruth,Eschstruth,Nataly von,Der Majoratsherr. II. Band,NaN,NaN,1898,https://www.projekt-gutenberg.org/eschstru/maj...,eschstru,Q87140,...,53190,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:36,2025-11-16T00:46:36,2025-11-16T00:46:36
661,Nataly von Eschstruth,Eschstruth,Nataly von,Frieden I,NaN,NaN,1905,NaN,NaN,Q87140,...,58170,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:37,2025-11-16T00:46:37,2025-11-16T00:46:37
664,Nataly von Eschstruth,Eschstruth,Nataly von,Jung gefreit - 2,NaN,NaN,1897,NaN,NaN,Q87140,...,57365,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:39,2025-11-16T00:46:39,2025-11-16T00:46:39


In [ ]:
print('\n'.join(view.loc[~sel, 'title']))

In [197]:
for row in view.itertuples():
    if row in view[sel]:
        print('Works!')
        break

In [191]:
view[view['title'].str.contains('Er und Sie')]

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,1e_qid_by,work_qid_date,ed1_qid_date,pgde_qid_date
index,,,,,,,,,,,,,,,,,,,,,
2359,Charlotte Niese,Niese,Charlotte,Er und Sie und andere Novellen,NaN,NaN,1925,NaN,NaN,Q74303,...,15614,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:47:34,2025-11-16T00:47:34,2025-11-16T00:47:34


In [186]:
view[view['title'].str.contains('Frieden')]

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,1e_qid_by,work_qid_date,ed1_qid_date,pgde_qid_date
index,,,,,,,,,,,,,,,,,,,,,
653,Nataly von Eschstruth,Eschstruth,Nataly von,Frieden II,NaN,NaN,1905,NaN,NaN,Q87140,...,52244,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:31,2025-11-16T00:46:31,2025-11-16T00:46:31
661,Nataly von Eschstruth,Eschstruth,Nataly von,Frieden I,NaN,NaN,1905,NaN,NaN,Q87140,...,58170,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:37,2025-11-16T00:46:37,2025-11-16T00:46:37


# Upload them!

In [ ]:
login_instance = wbi_login.OAuth2(consumer_token=CONSUMER_TOKEN, 
                                  consumer_secret=CONSUMER_SECRET, 
                                  mediawiki_api_url=wbi_config['MEDIAWIKI_API_URL'])
wbi = WikibaseIntegrator(login=login_instance)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'{BOTNAME}: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'

### This is an older, less preferred method of log-in.

In [ ]:
login_instance = wbi_login.Login(user=USER_AUTH, password=PASSWORD, user_agent=wbi_config['USER_AGENT'], 
                                 mediawiki_api_url=wbi_config['MEDIAWIKI_API_URL'])

wbi = WikibaseIntegrator(login=login_instance)

randhex = "{:x}".format(random.randrange(0, 2**48))
EDIT_SUMMARY=f'{BOTNAME}: test ([[:toolforge:editgroups/b/CB/{randhex}|details]])'

In [123]:
def date_tag_precise():
    return datetime.today().strftime("%Y-%m-%dT%H:%M:%S")

class Author():
    def __init__(self, item = None, qid=None, name = None, works = None, editions = None):
        self._item = item
        self._qid = qid
        self._name = name
        self._works = works
        self._editions = editions
    
    @classmethod
    def from_item(self, item: wikibaseintegrator.entities.item.ItemEntity):
        return Author(item=item)
    
    def get_name(self, language='mul'):
        if self._item:
            if self._item.labels.get(language):
                return self._item.labels.get(language).value
            if self._item.labels.get('mul'):
                return self._item.labels.get('mul').value
            if self._name:
                return self._name
            else:
                if self._quid:
                    raise Exception(f'No name available for {self._qid}')
                if self.item:
                    raise Exception(f'No name available for {self._item}')
                raise Exception(f'No name available for {self}')
        else:
            return self.name
        
def format_year(year : int):
    return datetime(year, 1, 1).strftime("+%Y-%m-%dT%H:%M:%SZ")

def make_work(author_qid : str, author : Author, title : str, 
              year : int = None, url : Optional[str] = None): 
    language = ENTITIES['German']

    formatted_year = format_year(year)

    new_work = wbi.item.new()

    new_work.labels.set('de', title)
    # Set a default label too
    new_work.labels.set('mul', title)
    new_work.descriptions.set('en', f'Literary work of fiction by {author.get_name('en')}')
    new_work.descriptions.set('de', f'Fiktionales literarisches Werk von {author.get_name('de')}')
    
    new_work.claims.add([
        datatypes.Item(value=ENTITIES['literary_work'], prop_nr=PROPS['instance_of']), 
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        #
        # NEED TO CHECK IF IT HAS OTHER AUTHORS
        #
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                    ])
    if url:
        new_work.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])
    return new_work



def make_edition(author_qid : str, title : str, author : Author, year :int, 
                work_qid : Optional[str], url : Optional[str] = None): 
    language = ENTITIES['German']

    new_edition = wbi.item.new()
    new_edition.labels.set('de', f'{title} (Erstausgabe von {str(year)})')
    # Set a default label too
    new_edition.labels.set('mul', f'{title} (first edition, {str(year)})')

    new_edition.descriptions.set('en', 
                        f'{str(year)} edition of the literary work of fiction by {author.get_name('en')}')
    new_edition.descriptions.set('de', 
                        f'Ausgabe von {str(year)} des fiktionalen literarischen Werks von {author.get_name('de')}')

    formatted_year = format_year(year)
    
    new_edition.claims.add([
        datatypes.Item(value=ENTITIES['edition'], prop_nr=PROPS['instance_of']), 
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                            ])   

    if work_qid: 
        new_edition.claims.add([ datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of'])
                               ])
        
    if url:
        new_edition.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])

    return new_edition


def make_gutenberg_edition(author_qid : str, title : str, author : Author, source : str,
                           work_qid : Optional[str] = None, edition_qid : Optional[str] = None, 
                           url : Optional[str] = None, year : Optional[int]=None, pg_id : Optional[str] = None): 

    language = ENTITIES['German']

    new_edition = wbi.item.new()

    if source == 'PG-DE':
        gb_edition_descr = {'en': 'Projekt Gutenberg-DE edition', 
                          'de': 'Projekt Gutenberg-DE Edition'}
    elif source == 'PG-US':
        gb_edition_descr = {'en': 'Project Gutenberg edition', 
                          'de': 'Project Gutenberg Edition'}
    else:
        raise Exception(f"source must be one of 'PG-DE' or 'PG-US'")

    new_edition.labels.set('de', title + f' ({gb_edition_descr['de']})')
    # Set a default label too
    new_edition.labels.set('mul', title + f' ({gb_edition_descr['en']})')

    new_edition.descriptions.set('en', 
                f'{gb_edition_descr['en']} of the literary work of fiction by {author.get_name('de')}')
    new_edition.descriptions.set('de', 
                f'{gb_edition_descr['de']} des fiktionalen literarischen Werks von {author.get_name('de')}')

    new_edition.claims.add([
        datatypes.Item(value=ENTITIES['edition'], prop_nr=PROPS['instance_of']), 
        datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of']),
        datatypes.MonolingualText(text=title, language='de', prop_nr=PROPS['title']),
        datatypes.Item(value=author_qid, prop_nr=PROPS['author']),
        # We're not sure about the year, so let's leave it out
        #datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
        datatypes.Item(value=language, prop_nr=PROPS['language'])
                            ])

    if work_qid: 
        new_edition.claims.add([ datatypes.Item(value=work_qid, prop_nr=PROPS['edition_of'])
                               ])
        
    if edition_qid: 
        new_edition.claims.add([ datatypes.Item(value=edition_qid, prop_nr=PROPS['based_on'])
                               ])   
    if url:
        new_edition.claims.add([
            datatypes.URL(value=url, prop_nr=PROPS['work_available_at_URL'])
                        ])
        
    if year:
        formatted_year = format_year(year)
        new_edition.claims.add([
            datatypes.Time(time=formatted_year, precision=9, prop_nr=PROPS['publication_date']),
                        ])
                  # already made sure its a string in the function header, but let's be robust
    if pg_id and (source == 'PG-US'):
            new_edition.claims.add([
                datatypes.ExternalID(value=pg_id, prop_nr=PROPS['project_gb_ebook_id']),
                        ])
        
    return new_edition




def add_editions_to_work(work_qid : str, edition_qids : list[str]):
    work = wbi.item.get(entity_id=work_qid)
    claims_to_add = [datatypes.Item(value=edition_qid, prop_nr=PROPS['has_edition_or_translation']) 
                     for edition_qid in edition_qids]
    work.claims.add(claims_to_add)
    work.write()


### Time to actually do it

In [199]:
view = metadata[metadata['author_qid'].isin(author_qids_no_works) & (~select_part_of_works)]
view

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,1e_qid_by,work_qid_date,ed1_qid_date,pgde_qid_date
index,,,,,,,,,,,,,,,,,,,,,
19,Luise Ahlborn,Ahlborn,Luise,Schloß Favorite,NaN,NaN,1887,https://www.projekt-gutenberg.org/ahlborn/favo...,ahlborn,Q1324050,...,77373,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:07,2025-11-16T00:46:07,2025-11-16T00:46:07
124,Brigitte Augusti,Augusti,Brigitte,Mädchenlose,NaN,NaN,1888,https://www.projekt-gutenberg.org/augusti/maed...,augusti,Q19144479,...,43187,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:08,2025-11-16T00:46:08,2025-11-16T00:46:08
192,Käthe van Beeker,Beeker,Käthe van,Der Ring der Nuramaja,NaN,NaN,1919,https://www.projekt-gutenberg.org/beeker/ringn...,beeker,Q1795559,...,62196,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:09,2025-11-16T00:46:09,2025-11-16T00:46:09
197,Alice Berend,Berend,Alice,Frau Hempels Tochter,NaN,NaN,1912,NaN,NaN,Q2646840,...,47754,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:10,2025-11-16T00:46:10,2025-11-16T00:46:10
198,Alice Berend,Berend,Alice,Die Bräutigame der Babette Bomberling,NaN,NaN,1915,https://www.projekt-gutenberg.org/berend/bombe...,berend,Q2646840,...,32898,PG-DE,None,None,NaN,NaN,NaN,2025-11-16T00:46:10,2025-11-16T00:46:10,2025-11-16T00:46:10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4130,Anna Schieber,Schieber,Anna,Alle guten Geister...: Roman,NaN,54469,1905,NaN,NaN,Q563171,...,121841,PG-US,None,None,NaN,NaN,NaN,2025-11-16T00:48:29,2025-11-16T00:48:29,2025-11-16T00:48:29
4131,Anna Schieber,Schieber,Anna,Heimat: Erzählungen,NaN,67701,1915,NaN,NaN,Q563171,...,37457,PG-US,None,None,NaN,NaN,NaN,2025-11-16T00:48:30,2025-11-16T00:48:30,2025-11-16T00:48:30
4132,Anna Schieber,Schieber,Anna,Ludwig Fugeler: Roman,NaN,34424,1918,NaN,NaN,Q563171,...,98317,PG-US,None,None,NaN,NaN,NaN,2025-11-16T00:48:31,2025-11-16T00:48:31,2025-11-16T00:48:31


In [ ]:
# TODO: There is warning here, meaning we forget to set dtype on column first:
# /var/folders/ck/dw0r0d9x6v9d5y4pgvkx70x80000gn/T/ipykernel_38899/257793778.py:70: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Q136801459' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.


for work in view.itertuples():
    for col in ['author_qid', 'author', 'title', 'year', 'source']:
        if pd.isna(getattr(work, col)) or (getattr(work, col) == 'nan'):
            msg = f'Has NaN in metadata, skipping: {i} col: {col}'
            print(msg)
            logger.warning(msg)
            continue
    if pd.isna(work.url): #or (work.url == 'nan'):
        url = None
    else: 
        url = work.url.strip()

    if pd.isna(work.gutenberg_id): # or (work.source == 'nan'):
        pg_id = None
    else: 
        pg_id = work.gutenberg_id.strip()

    author_qid = work.author_qid.strip()
    author_entity = wbi.item.get(entity_id=author_qid)
    author = Author.from_item(author_entity)
    title = work.title.strip()
    formatted_year = format_year(work.year)
    source = work.source.strip()

    if title == 'Schloß Favorite':
        continue

    logger.info(
        f'{work[0]}, {author_qid}, {work.author}, {title}, {work.year}, {source}, {url}, {pg_id}'
        )

    

    new_work = make_work(author_qid = author_qid, author=author, title = title, 
                    year = work.year, url=url)
    
    new_work.write(login_instance=login_instance, summary=EDIT_SUMMARY)
    logger.info(new_work.get_json())
    work_qid = new_work.id

    metadata.at[work[0], 'work_qid_by'] = work_qid
    metadata.at[work[0], 'work_qid_date'] = date_tag_precise()

    new_edition = make_edition(author_qid = author_qid, year = work.year,
                           author = author,
                        title = title, 
                        work_qid = work_qid)
    new_edition.write(login_instance=login_instance, summary=EDIT_SUMMARY)
    logger.info(new_edition.get_json())
    edition_qid = new_edition.id

    metadata.at[work[0], 'ed1_qid_by'] = edition_qid
    metadata.at[work[0], 'ed1_qid_date'] = date_tag_precise()

    new_pg_edition = make_gutenberg_edition(author_qid = author_qid, 
                           title = title, 
                           author = author, 
                           work_qid= work_qid,
                           edition_qid = edition_qid, 
                           pg_id=pg_id, 
                           source=source,
                           url=url)
    new_pg_edition.write(login_instance=login_instance, summary=EDIT_SUMMARY)
    logger.info(new_pg_edition.get_json())
    pg_edition_qid = new_pg_edition.id
    if source == 'PG-DE':
        metadata.at[work[0], 'pgde_qid_by'] = work_qid
        metadata.at[work[0], 'pgde_qid_date'] = date_tag_precise()
    elif source == 'PG-US':
        metadata.at[work[0], 'pgus_qid_by'] = work_qid
        metadata.at[work[0], 'pgus_qid_date'] = date_tag_precise()

    add_editions_to_work(work_qid=work_qid, edition_qids=[edition_qid, pg_edition_qid])

/var/folders/ck/dw0r0d9x6v9d5y4pgvkx70x80000gn/T/ipykernel_38899/257793778.py:70: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Q136801459' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  metadata.at[work[0], 'pgus_qid_by'] = work_qid


In [226]:
metadata[metadata['author'] == 'Felicitas Rose']

,author,author_last,author_first,title,work_qid,gutenberg_id,year,url,pgde_author_id,author_qid,...,num_tokens,source,work_qid_by,pgde_qid_by,pgus_qid_by,ed1_qid_by,work_qid_date,ed1_qid_date,pgde_qid_date,pgus_qid_date
index,,,,,,,,,,,,,,,,,,,,,
2672,Felicitas Rose,Rose,Felicitas,Heideschulmeister Uwe Karsten,NaN,NaN,1909,https://www.projekt-gutenberg.org/rose/heidesc...,rose,Q1403083,...,62446,PG-DE,Q136801284,Q136801284,NaN,Q136801285,2025-11-16T03:41:03,2025-11-16T03:41:04,2025-11-16T03:41:05,NaN
2673,Felicitas Rose,Rose,Felicitas,Kerlchens Flitterwochen. Provinzmädel Band 8,NaN,NaN,1902,NaN,NaN,Q1403083,...,39444,PG-DE,None,None,NaN,NaN,NaN,NaN,NaN,NaN
2674,Felicitas Rose,Rose,Felicitas,Kleinstadtluft. Provinzmädel Band 1,NaN,NaN,1902,NaN,NaN,Q1403083,...,47184,PG-DE,None,None,NaN,NaN,NaN,NaN,NaN,NaN
2675,Felicitas Rose,Rose,Felicitas,Bilder aus den vier Wänden,NaN,NaN,1911,https://www.projekt-gutenberg.org/rose/bild4wa...,rose,Q1403083,...,79644,PG-DE,Q136801287,Q136801287,NaN,Q136801288,2025-11-16T03:41:07,2025-11-16T03:41:08,2025-11-16T03:41:08,NaN
2676,Felicitas Rose,Rose,Felicitas,Das Haus mit den grünen Fensterläden,NaN,NaN,1930,NaN,NaN,Q1403083,...,78241,PG-DE,Q136801290,Q136801290,NaN,Q136801291,2025-11-16T03:41:11,2025-11-16T03:41:11,2025-11-16T03:41:12,NaN
2677,Felicitas Rose,Rose,Felicitas,Kerlchens Mutterglück. Provinzmädel Band IX,NaN,NaN,1903,NaN,NaN,Q1403083,...,38758,PG-DE,None,None,NaN,NaN,NaN,NaN,NaN,NaN
2678,Felicitas Rose,Rose,Felicitas,Kerlchen als Anstandsdame,NaN,NaN,1903,https://www.projekt-gutenberg.org/rose/provinz...,rose,Q1403083,...,37461,PG-DE,Q136801293,Q136801293,NaN,Q136801294,2025-11-16T03:41:14,2025-11-16T03:41:15,2025-11-16T03:41:16,NaN
2679,Felicitas Rose,Rose,Felicitas,Kerlchens Lern- und Wanderjahre,NaN,NaN,1903,https://www.projekt-gutenberg.org/rose/provinz...,rose,Q1403083,...,41241,PG-DE,Q136801296,Q136801296,NaN,Q136801297,2025-11-16T03:41:18,2025-11-16T03:41:19,2025-11-16T03:41:19,NaN
2680,Felicitas Rose,Rose,Felicitas,Kerlchen als Erzieher. Provinzmädel Band 3,NaN,NaN,1903,NaN,NaN,Q1403083,...,34400,PG-DE,None,None,NaN,NaN,NaN,NaN,NaN,NaN
